# 💰 CampusNova AI — Expected CTC / Salary Package Regressor
### Model 2: Campus Compensation Prediction Pipeline (Repeated CV & Mean Baseline)
**Target:** CTC Salary Package in Lakhs Per Annum (LPA)  
**Dataset:** Kaggle `benroshan/factors-influencing-campus-placement` (`Placement_Data_Full_Class.csv` - 215 records) + `anmol4210/placement-data-full-class`  
**Direct Kaggle Link:** https://www.kaggle.com/datasets/benroshan/factors-influencing-campus-placement  
**Features:** Degree %, Employability Aptitude Test (E-Test) %, Work Experience  

---
### ⚠️ Scientific Domain Transparency Note (Point 7):
The Ben Roshan Campus Recruitment dataset represents institutional campus hiring from **MBA and undergraduate commerce/science/tech** degrees (`mba_p`, `specialisation`).
While it serves as an empirical compensation benchmark for campus placement aptitude and academic performance, it should **not be misrepresented as pure B.Tech engineering salaries**. We treat this model as an empirical campus recruitment compensation benchmark.

---
### Data Science Integrity Guarantees (Points 5, 8 & Small Things):
1. **No Dead Columns:** Eliminated unused zero arrays (`stream_code`).
2. **Small Sample Handling:** Tested on ~148 placed records using **Repeated K-Fold CV** (5 folds x 5 repeats) rather than noisy single splits.
3. **Baseline Comparison:** Benchmarked against a **DummyRegressor(strategy='mean')** baseline to verify true statistical signal.
4. **Unit Verification:** Verified raw salary in INR and scaled to Lakhs Per Annum (LPA).


### 📦 Step 1: Install Required Libraries


In [ ]:
!pip install -q kaggle scikit-learn pandas numpy joblib matplotlib seaborn
print("Dependencies installed!")


### 🔑 Step 2: Masked Kaggle Authentication (`getpass` or `kaggle.json`)


In [ ]:
import os, json, getpass
from google.colab import files

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kaggle_json_path = os.path.expanduser("~/.kaggle/kaggle.json")

print("Upload kaggle.json OR leave empty to input securely:")
uploaded = files.upload()

if "kaggle.json" in uploaded:
    with open(kaggle_json_path, "wb") as f:
        f.write(uploaded["kaggle.json"])
    os.chmod(kaggle_json_path, 0o600)
    print("Successfully authenticated with uploaded kaggle.json!")
else:
    u = input("Kaggle Username: ").strip()
    k = getpass.getpass("Enter Kaggle API Key (hidden): ").strip()
    if u and k:
        with open(kaggle_json_path, "w") as f:
            json.dump({"username": u, "key": k}, f)
        os.chmod(kaggle_json_path, 0o600)
        print("Successfully authenticated with masked credentials!")


### 📥 Step 3: Resilient Dataset Ingestion (Kaggle + Multi-Mirror Fallback)


In [ ]:
import os, urllib.request
import pandas as pd
import numpy as np

os.makedirs("data/raw", exist_ok=True)
os.makedirs("models", exist_ok=True)

target_csv = "data/raw/Placement_Data_Full_Class.csv"
alt_csv = "data/raw/Placement_Data.csv"

# Tier 1: Primary Kaggle download (slug: benroshan/factors-affecting-campus-placement)
if not os.path.exists(target_csv) and not os.path.exists(alt_csv):
    print("Attempting Kaggle CLI download (benroshan/factors-affecting-campus-placement)...")
    !kaggle datasets download -d benroshan/factors-affecting-campus-placement -p data/raw --unzip || true

# Tier 2: Alternate Kaggle mirror slug (anmol4210/placement-data-full-class)
if not os.path.exists(target_csv) and not os.path.exists(alt_csv):
    print("Trying alternate Kaggle slug (anmol4210/placement-data-full-class)...")
    !kaggle datasets download -d anmol4210/placement-data-full-class -p data/raw --unzip || true

# Tier 3: Direct verified HTTPS Raw Mirror (Zero authentication required, solves 403 Forbidden)
if not os.path.exists(target_csv) and not os.path.exists(alt_csv):
    print("Kaggle returned 403 Forbidden. Downloading directly from verified public GitHub mirror...")
    mirror_urls = [
        "https://raw.githubusercontent.com/ShuklaPrashant21/Campus_Recruitment/master/Placement_Data_Full_Class.csv",
        "https://raw.githubusercontent.com/shubham070696/Placement_Factor_Analysis/master/Placement_Data_Full_Class.csv",
        "https://raw.githubusercontent.com/shahil04/placement_analysis/master/Placement_Data_Full_Class.csv"
    ]
    for url in mirror_urls:
        try:
            print(f"Fetching from: {url}")
            urllib.request.urlretrieve(url, target_csv)
            if os.path.exists(target_csv) and os.path.getsize(target_csv) > 1000:
                print("✓ Successfully downloaded Placement_Data_Full_Class.csv from public mirror!")
                break
        except Exception as e:
            print(f"Mirror fetch failed ({e}), trying next...")

# Tier 4: Manual upload fallback if network is completely blocked
if not os.path.exists(target_csv) and not os.path.exists(alt_csv):
    from google.colab import files
    print("Please upload Placement_Data_Full_Class.csv manually:")
    up = files.upload()
    for fname in up:
        if fname.endswith(".csv"):
            os.rename(fname, target_csv)
            break

# Flexible path resolution
actual_csv = target_csv if os.path.exists(target_csv) else alt_csv
df_raw = pd.read_csv(actual_csv)
print(f"Total applicants loaded from {actual_csv}: {len(df_raw)}")

# Filter placed candidates and convert Salary in INR to LPA
df_placed = df_raw[df_raw["status"] == "Placed"].copy()
df_placed["salary_lpa"] = (df_placed["salary"] / 100000.0).round(2)
df_placed["workex_binary"] = (df_placed["workex"] == "Yes").astype(int)

# Check duplicates
print(f"Duplicate placed rows: {df_placed.duplicated().sum()}")
print(f"Placed rows for regression: {len(df_placed)} records")
print(f"Salary LPA Range: {df_placed['salary_lpa'].min():.2f} to {df_placed['salary_lpa'].max():.2f} LPA (Mean: {df_placed['salary_lpa'].mean():.2f} LPA)")
display(df_placed[["degree_p", "etest_p", "workex_binary", "salary_lpa"]].head(3))


### 🛠️ Step 4: Pipeline Assembly & Repeated K-Fold Cross Validation (Point 8)
We define a clean `ColumnTransformer` (no dead columns) and evaluate candidates against a dummy baseline using `RepeatedKFold(5, 5)`.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, RepeatedKFold, cross_validate
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# Features: degree_p, etest_p, workex_binary (no dead stream_code)
X = df_placed[["degree_p", "etest_p", "workex_binary"]]
y = df_placed["salary_lpa"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training split: {len(X_train)} | Holdout test split: {len(X_test)}")

pre_sal = ColumnTransformer([
    ("num", StandardScaler(), ["degree_p", "etest_p"]),
    ("keep", "passthrough", ["workex_binary"])
])

regressors = {
    "Baseline (Predict Mean)": DummyRegressor(strategy="mean"),
    "Baseline (Predict Median)": DummyRegressor(strategy="median"),
    "Ridge Regression (alpha=1.0)": Ridge(alpha=1.0, random_state=42),
    "Linear Regression (OLS)": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, random_state=42)
}

rkf = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
cv_summary = {}
best_cv_mae = float("inf")
best_reg_name = ""

print("-" * 75)
print(f"{'Salary Regressor Model':<32} | {'Repeated CV MAE':<16} | {'CV R2 Score':<12}")
print("-" * 75)

for name, reg in regressors.items():
    pipe = Pipeline([("pre", pre_sal), ("reg", reg)])
    scores = cross_validate(pipe, X_train, y_train, cv=rkf, scoring=["neg_mean_absolute_error", "r2"])
    mean_mae = -float(np.mean(scores["test_neg_mean_absolute_error"]))
    mean_r2 = float(np.mean(scores["test_r2"]))
    
    cv_summary[name] = {"cv_mae": round(mean_mae, 4), "cv_r2": round(mean_r2, 4)}
    print(f"{name:<32} | {mean_mae:.4f} LPA         | {mean_r2:.4f}")
    
    if "Baseline" not in name and mean_mae < best_cv_mae:
        best_cv_mae = mean_mae
        best_reg_name = name

print("-" * 75)
print(f"Selected Winning Regressor via Repeated CV: '{best_reg_name}' (CV MAE: {best_cv_mae:.4f} LPA)")


### 📊 Step 5: Final Unbiased Holdout Evaluation & Baseline Comparison


In [ ]:
winner_reg_pipe = Pipeline([
    ("pre", pre_sal),
    ("reg", regressors[best_reg_name])
])
winner_reg_pipe.fit(X_train, y_train)

y_pred = winner_reg_pipe.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_r2 = r2_score(y_test, y_pred)

dummy_mae = mean_absolute_error(y_test, np.full_like(y_test, np.mean(y_train)))

print("=" * 60)
print(f"UNBIASED TEST EVALUATION FOR {best_reg_name.upper()}:")
print("=" * 60)
print(f"Test MAE:       {test_mae:.4f} LPA")
print(f"Baseline MAE:   {dummy_mae:.4f} LPA (Mean Baseline)")
print(f"MAE Reduction:  {((dummy_mae - test_mae) / dummy_mae) * 100:.1f}% improvement")
print(f"Test RMSE:      {test_rmse:.4f} LPA")
print(f"Test R2 Score:  {test_r2:.4f}")


### 📦 Step 6: Export Production Artifacts & Auto-Download


In [ ]:
import zipfile
from google.colab import files

joblib.dump(winner_reg_pipe, "models/salary_pipeline.joblib")
with open("models/regressor_benchmarks.json", "w") as f:
    json.dump({
        "production_model": best_reg_name,
        "best_cv_mae_lpa": best_cv_mae,
        "test_metrics": {"test_mae_lpa": test_mae, "dummy_baseline_mae": dummy_mae, "test_r2": test_r2},
        "cv_leaderboard": cv_summary,
        "dataset_note": "Ben Roshan Campus Recruitment benchmark dataset (MBA/Undergrad records)."
    }, f, indent=2)

zip_name = "campusnova_salary_model.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("models/salary_pipeline.joblib", "salary_pipeline.joblib")
    z.write("models/regressor_benchmarks.json", "regressor_benchmarks.json")
    z.write("data/raw/Placement_Data_Full_Class.csv", "data/raw/Placement_Data_Full_Class.csv")

print(f"Archive ready: {zip_name}")
files.download(zip_name)
print("Pipeline downloaded! Place it in placepulse_ds_lab/models/.")
